# DINGO poster plots

This notebook is the extracted poster-plot section from `fit-kinematics-mcmc.ipynb`. It rebuilds the upstream fit state from saved Sérsic/MCMC products, skips expensive morphology, MCMC, and mock reruns, and then exposes the poster plotting cells below for independent editing.

Run from this directory with the DINGO kernel. The setup cell intentionally reads the source notebook for shared data preparation and model definitions; the poster cells themselves are copied here.


In [ ]:
from pathlib import Path
import json

SOURCE_NOTEBOOK = Path('fit-kinematics-mcmc.ipynb')
if not SOURCE_NOTEBOOK.exists():
    raise FileNotFoundError(f'Cannot find {SOURCE_NOTEBOOK.resolve()}')
source_nb = json.loads(SOURCE_NOTEBOOK.read_text())
ns = globals()

# Recreate the lightweight upstream state needed by the poster cells.
# Cells 13--14 (two-component morphology sampling) and 45--47 (batch runs)
# are intentionally skipped; their saved products are loaded by the poster cells.
setup_cells = [1, 2, 3, 5, 6, 7, 9, 11, 12, 15, 16, 18, 19,
               27, 29, 31, 33, 35, 37, 38, 39,
               41, 42, 43, 44]
for _cell_idx in setup_cells:
    if source_nb['cells'][_cell_idx].get('cell_type') == 'markdown':
        continue
    _cell_source = ''.join(source_nb['cells'][_cell_idx].get('source', []))
    if _cell_idx == 1:
        _cell_source = _cell_source.replace('RUN_PYSERSIC       = True', 'RUN_PYSERSIC       = False')
        _cell_source = _cell_source.replace('RUN_MCMC           = True', 'RUN_MCMC           = False')
        _cell_source = _cell_source.replace('RUN_MCMC           = False', 'RUN_MCMC           = False')
        _cell_source = _cell_source.replace('RUN_FLOOR_TEST     = False', 'RUN_FLOOR_TEST     = False')
    _cell_source = '\n'.join(line for line in _cell_source.splitlines()
                              if not line.lstrip().startswith('%'))
    exec(compile(_cell_source, f'{SOURCE_NOTEBOOK}:cell {_cell_idx}', 'exec'), ns, ns)

# Load an already-generated chain; never launch a production chain from this
# plotting notebook. Run fit-kinematics-mcmc.ipynb first if this file is absent.
if not Path(CHAIN).exists():
    raise FileNotFoundError(
        f'No fitted chain at {CHAIN}. Run fit-kinematics-mcmc.ipynb first.')
saved = np.load(CHAIN, allow_pickle=True)
chain = saved['chain']
names = [str(s) for s in saved['param_names']]
nburn = int(saved['nburn'])
flat = chain[nburn:].reshape(-1, chain.shape[-1])
post = {n: float(np.percentile(flat[:, j], 50)) for j, n in enumerate(names)}

# Recreate the retained headline state that the poster cells consume.
for _cell_idx in (48, 49):
    if source_nb['cells'][_cell_idx].get('cell_type') == 'markdown':
        continue
    _cell_source = ''.join(source_nb['cells'][_cell_idx].get('source', []))
    _cell_source = '\n'.join(line for line in _cell_source.splitlines()
                              if not line.lstrip().startswith('%'))
    exec(compile(_cell_source, f'{SOURCE_NOTEBOOK}:cell {_cell_idx}', 'exec'), ns, ns)

print('Poster setup complete; edit/run the extracted cells below.')


## Poster figures

The full 11x11 corner is a diagnostic, not a figure -- six of its parameters are
nuisances (two grism offsets each, the two centre coordinates, the noise scale) and
nobody reads them on a poster. Two smaller ones carry the story:

**`corner_science`** -- `V_rot`, `i`, `R_v`. The banana between the first two IS the
result: the grism measures `V*sin(i)`, so `V_rot` and `i` trade off along a ridge and
only their product is pinned. This is the figure that explains why an inclination is
needed at all.

**`corner_derived`** -- `V*sin(i)`, `R_v` in kpc, `log M_dyn(<5 kpc)`. The physical
answers, in units a reader cares about, with the correlations between them intact --
which is what you lose if you quote three numbers with separate error bars.

In [ ]:
import corner

def poster_corner(cols, labels, fname, truths=None, title=None):
    data = np.column_stack(cols)
    fig = corner.corner(
        data, labels=labels, truths=truths,
        show_titles=True, title_fmt='.1f', quantiles=[0.16, 0.5, 0.84],
        title_kwargs={'fontsize': 13}, label_kwargs={'fontsize': 14},
        hist_kwargs={'lw': 1.6}, plot_datapoints=False, fill_contours=True,
        levels=(0.393, 0.865),          # 1 and 2 sigma in 2D, not 68/95
        contour_kwargs={'linewidths': 1.2})
    for ax in fig.get_axes():
        ax.grid(False)
        ax.tick_params(labelsize=11)
    if title:
        fig.suptitle(title, fontsize=15, y=1.02)
    fig.savefig(out(fname), dpi=200, bbox_inches='tight')
    plt.show()
    return fig

# --- the degeneracy: this is the physics figure -------------------------------
poster_corner(
    [flat[:, iV], np.rad2deg(flat[:, iI]), flat[:, iR]*KPC_PER_PIX],
    [r'$V_{\rm rot}$ [km s$^{-1}$]', r'$i$ [deg]', r'$R_v$ [kpc]'],
    'corner_science.png',
    title=f'ID {ID}   $z={z:.4f}$')

# --- the answers ---------------------------------------------------------------
_vsini = flat[:, iV]*np.sin(flat[:, iI])
_rv    = flat[:, iR]*KPC_PER_PIX
_logm  = np.log10(arctangent_dynamical_mass(5.0, flat[:, iV], _rv))
poster_corner(
    [_vsini, _rv, _logm],
    [r'$V_{\rm rot}\sin i$ [km s$^{-1}$]', r'$R_v$ [kpc]',
     r'$\log M_{\rm dyn}(<5\,{\rm kpc})$'],
    'corner_derived.png',
    title=f'ID {ID}   derived quantities')

### Vertical (4x2) version of the R/C panels

Same figure as `plot.plot_kinematics_fitting_result` produces, transposed: the four
quantities run down the page and the two grisms sit side by side. Tall and narrow,
which fits a poster column.

No new plotting code -- that function takes a sequence of four axes and never cares
whether they came from a row or a column, so passing `axs[:, 0]` instead of `axs[0]`
is the whole change.

In [ ]:
def panels_vertical(imR, imC, vR, vC, velp, dR, dC, tag, fname,
                    figsize=(7.2, 14.0), rows=('image', 'model', 'residual', 'velocity')):
    """R and C down two columns, styled to match the existing poster figure:
    axes in arcsec measured from the cutout corner (0 -> ~5"), one colourbar per
    panel on its own scale, full frame shown (nothing masked out).

    Drawn directly rather than through plot.plot_kinematics_fitting_result, which
    imshow()s without an `extent` and marks the centre in pixel coordinates.
    """
    s = arcsec_per_pixel                       # 0.063 "/pix, NIRCam LW
    imR, imC, vR, vC = map(_np, (imR, imC, vR, vC))
    rawR, rawC = _np(true_grism_R), _np(true_grism_C)
    resid = imR - imC
    x0, y0 = float(velp['x0_v']), float(velp['y0_v'])

    spec = {
        'image':    lambda g: (rawR if g == 'R' else rawC, f'{g} Grism Image',   None, None),
        'model':    lambda g: (imR  if g == 'R' else imC,  f'{g} Best-fit model', None, None),
        'residual': lambda g: ((+1 if g == 'R' else -1)*resid, f'{g} Residual', 'seismic', 'sym'),
        'velocity': lambda g: (vR if g == 'R' else vC, f'{g} velocity field', 'seismic', 'sym'),
    }

    fig, axs = plt.subplots(len(rows), 2, figsize=figsize, squeeze=False)
    for col, (g, dsp) in enumerate([('R', dR), ('C', dC)]):
        # the raw grism frame is offset from the rectified one by the fitted dx/dy
        cx, cy = x0 - float(dsp['dx']), y0 - float(dsp['dy'])
        for row, key in enumerate(rows):
            img, title, cmap, scale = spec[key](g)
            ax = axs[row, col]
            ny, nx = img.shape
            kw = dict(origin='lower', extent=[0, nx*s, 0, ny*s])
            if cmap:
                lim = np.nanmax(np.abs(img))
                kw.update(cmap=cmap, vmin=-lim, vmax=lim)
            m = ax.imshow(img, **kw)
            mx, my = ((cx if key == 'image' else x0) + 0.5)*s, \
                     ((cy if key == 'image' else y0) + 0.5)*s
            ax.plot(mx, my, marker='+', ms=10, mew=1.5, color='lime')
            ax.set_title(title, fontsize=12, pad=5)
            ax.set_xlabel('Arcsec', fontsize=11)
            ax.set_ylabel('Arcsec', fontsize=11)
            ax.tick_params(labelsize=10)
            ax.grid(False)
            cb = fig.colorbar(m, ax=ax, fraction=0.046, pad=0.03)
            cb.ax.tick_params(labelsize=9)

    plt.tight_layout()
    fig.savefig(out(fname), dpi=200, bbox_inches='tight')
    plt.show()
    print(f'saved -> {out(fname)}')
    return fig


# --- Adam MAP -----------------------------------------------------------------
panels_vertical(image_R, image_C, vz_R, vz_C,
                velocity_params, dispersion_params_R, dispersion_params_C,
                'Adam MAP', 'poster_panels_vertical_adam.png')

# --- MCMC posterior median ------------------------------------------------------
panels_vertical(image_R_mc, image_C_mc, vz_R_mc, vz_C_mc,
                vel_mc, disp_R_mc, disp_C_mc,
                'MCMC posterior median', 'poster_panels_vertical_mcmc_median.png')

# three rows only (drop the velocity field), matching the existing poster panel:
# panels_vertical(image_R, image_C, vz_R, vz_C, velocity_params,
#                 dispersion_params_R, dispersion_params_C, 'Adam MAP',
#                 'poster_panels_3row.png', figsize=(7.2, 10.5),
#                 rows=('image', 'model', 'residual'))

### Horizontal (2x4) version, arcsec axes

The original two-row layout -- R across the top, C across the bottom -- with the axes
in arcsec instead of pixels. Nothing else changed: same panels, same per-panel colour
scales, same full frame.

In [ ]:
def panels_horizontal(imR, imC, vR, vC, velp, dR, dC, tag, fname,
                      figsize=(19.0, 9.0)):
    """2 rows (R, C) x 4 columns (grism image / best-fit model / residual /
    velocity field), axes in arcsec measured from the cutout corner."""
    s = arcsec_per_pixel                       # 0.063 "/pix, NIRCam LW
    imR, imC, vR, vC = map(_np, (imR, imC, vR, vC))
    rawR, rawC = _np(true_grism_R), _np(true_grism_C)
    resid = imR - imC
    x0, y0 = float(velp['x0_v']), float(velp['y0_v'])

    fig, axs = plt.subplots(2, 4, figsize=figsize)
    for row, (g, raw, mod, vz, dsp, sign) in enumerate(
            [('R', rawR, imR, vR, dR, +1.0), ('C', rawC, imC, vC, dC, -1.0)]):
        cx, cy = x0 - float(dsp['dx']), y0 - float(dsp['dy'])   # raw grism frame
        cols = [(raw,        f'{g} Grism Image',    None,      (cx, cy)),
                (mod,        f'{g} Best-fit model', None,      (x0, y0)),
                (sign*resid, f'{g} Residual',       'seismic', (x0, y0)),
                (vz,         f'{g} velocity field', 'seismic', (x0, y0))]
        for col, (img, title, cmap, (mx, my)) in enumerate(cols):
            ax = axs[row, col]
            ny, nx = img.shape
            kw = dict(origin='lower', extent=[0, nx*s, 0, ny*s])
            if cmap:
                lim = np.nanmax(np.abs(img))
                kw.update(cmap=cmap, vmin=-lim, vmax=lim)
            m = ax.imshow(img, **kw)
            ax.plot((mx + 0.5)*s, (my + 0.5)*s, marker='+', ms=10, mew=1.5, color='lime')
            ax.set_title(title, fontsize=12, pad=5)
            ax.set_xlabel('Arcsec', fontsize=11)
            ax.set_ylabel('Arcsec', fontsize=11)
            ax.tick_params(labelsize=10)
            ax.grid(False)
            cb = fig.colorbar(m, ax=ax, fraction=0.046, pad=0.03)
            cb.ax.tick_params(labelsize=9)

    plt.tight_layout()
    fig.savefig(out(fname), dpi=200, bbox_inches='tight')
    plt.show()
    print(f'saved -> {out(fname)}')
    return fig


panels_horizontal(image_R, image_C, vz_R, vz_C,
                  velocity_params, dispersion_params_R, dispersion_params_C,
                  'Adam MAP', 'poster_panels_horizontal_adam.png')

panels_horizontal(image_R_mc, image_C_mc, vz_R_mc, vz_C_mc,
                  vel_mc, disp_R_mc, disp_C_mc,
                  'MCMC posterior median', 'poster_panels_horizontal_mcmc_median.png')

### Compact 2x4 for the poster column

All eight panels kept, but the space they waste is reclaimed:

- **axis labels only on the outer edges** -- one `Arcsec` on the bottom row and one on
  the left column instead of sixteen
- **one colourbar per column** spanning both rows, instead of eight. R and C then share
  a scale within a column, which is also more honest: the two grisms really do have
  different peak fluxes and separate scales hide that
- **larger fonts**, since the figure is scaled down to a 36 cm column
- tight `GridSpec` spacing

`share_cbar='panel'` reverts to eight independent colourbars if you prefer each panel
on its own scale.

In [ ]:
def panels_compact(imR, imC, vR, vC, velp, dR, dC, fname,
                   figsize=(21.0, 9.6), fs_title=21, fs_label=19, fs_tick=16):
    """2 rows (R, C) x 4 columns, laid out for a narrow poster column.

    All eight panels kept. Space is reclaimed by grouping the colourbars: the
    grism image and the best-fit model are the same quantity, so they share ONE
    bar on ONE scale -- which is also the honest comparison, since a model shown
    on its own stretch always looks like a good fit. Three bars, not eight, and
    none sitting between two adjacent image panels.
    """
    from matplotlib.gridspec import GridSpec
    import matplotlib.ticker as mticker
    s = arcsec_per_pixel
    imR, imC, vR, vC = map(_np, (imR, imC, vR, vC))
    rawR, rawC = _np(true_grism_R), _np(true_grism_C)
    resid = imR - imC
    x0, y0 = float(velp['x0_v']), float(velp['y0_v'])

    panels = [('Grism Image',    [rawR, rawC],    None),
              ('Best-fit model', [imR, imC],      None),
              ('Residual',       [resid, -resid], 'seismic'),
              ('velocity field', [vR, vC],        'seismic')]
    # which panel columns share a colourbar, and what that bar is labelled
    groups = [([0, 1], ''), ([2], ''), ([3], r'$v_z$ [km s$^{-1}$]')]

    # grid columns: image columns, with a narrow bar column after each GROUP
    widths, slot = [], {}
    for members, _ in groups:
        for c in members:
            slot[c] = len(widths); widths.append(1.0)
        widths.append(0.05)                       # the shared bar for this group
    fig = plt.figure(figsize=figsize)
    gs = GridSpec(2, len(widths), figure=fig, width_ratios=widths,
                  wspace=0.30, hspace=0.14,
                  left=0.05, right=0.985, top=0.94, bottom=0.10)

    for members, cblabel in groups:
        cmap = panels[members[0]][2]
        stack = [im for c in members for im in panels[c][1]]
        if cmap:
            lim = max(np.nanmax(np.abs(a)) for a in stack)
            vmin, vmax = -lim, lim
        else:
            vmin = min(np.nanmin(a) for a in stack)
            vmax = max(np.nanmax(a) for a in stack)

        first = None
        for c in members:
            name, imgs, _ = panels[c]
            for r, (g, img, dsp) in enumerate(zip(('R', 'C'), imgs, (dR, dC))):
                ax = fig.add_subplot(gs[r, slot[c]])
                ny, nx = img.shape
                kw = dict(origin='lower', extent=[0, nx*s, 0, ny*s],
                          vmin=vmin, vmax=vmax)
                if cmap:
                    kw['cmap'] = cmap
                m = ax.imshow(img, **kw)
                first = first or m
                cx, cy = ((x0 - float(dsp['dx']), y0 - float(dsp['dy']))
                          if name == 'Grism Image' else (x0, y0))
                ax.plot((cx + 0.5)*s, (cy + 0.5)*s, marker='+', ms=13, mew=2.0,
                        color='lime')
                ax.set_title(f'{g} {name}', fontsize=fs_title, pad=6)
                ax.tick_params(labelsize=fs_tick)
                ax.grid(False)
                if r == 1:
                    ax.set_xlabel('Arcsec', fontsize=fs_label)
                else:
                    ax.tick_params(labelbottom=False)
                if slot[c] == 0:
                    ax.set_ylabel('Arcsec', fontsize=fs_label)
                else:
                    ax.tick_params(labelleft=False)

        cax = fig.add_subplot(gs[:, slot[members[-1]] + 1])
        cb = fig.colorbar(first, cax=cax)
        cb.ax.tick_params(labelsize=fs_tick)
        cb.locator = mticker.MaxNLocator(nbins=5)
        cb.update_ticks()
        if cblabel:
            cb.set_label(cblabel, fontsize=fs_label - 3, labelpad=10)

    fig.savefig(out(fname), dpi=300, bbox_inches='tight')
    plt.show()
    print(f'saved -> {out(fname)}')
    return fig


panels_compact(image_R_mc, image_C_mc, vz_R_mc, vz_C_mc,
               vel_mc, disp_R_mc, disp_C_mc,
               'poster_panels_compact_mcmc_median.png')

### Circular speed + emission contours (poster pair)

Left: the **deprojected** circular speed `V_circ(r)`, i.e. what the disk is actually
doing, not the line-of-sight projection. Right: the Pa$\alpha$ emission-line map as
contours over the F444W continuum, showing where the gas is relative to the stars.

Both are masked to where the line is detected, so neither extrapolates the model into
empty sky.

In [ ]:
import scipy.ndimage as ndi


def connected_mask_from_center(mask, center_xy):
    """Keep only the blob connected to the centre pixel.

    Ported from 1.1-astr540-plot.ipynb. Without this a plain sigma cut leaves
    isolated specks of noise scattered around the frame, and the velocity model
    gets painted onto them.
    """
    if mask.dtype != bool:
        mask = mask.astype(bool)
    labeled, ncomp = ndi.label(mask)
    if ncomp == 0:
        return mask
    cx, cy = int(round(center_xy[0])), int(round(center_xy[1]))
    ny, nx = mask.shape
    lab = labeled[cy, cx] if (0 <= cy < ny and 0 <= cx < nx) else 0
    if lab == 0:                                   # centre not on a blob
        sizes = ndi.sum(mask, labeled, index=np.arange(1, ncomp + 1))
        lab = int(np.argmax(sizes)) + 1
    return labeled == lab


def poster_speed_and_contours(velp, fname='poster_speed_contours.png',
                              figsize=(16.0, 8.0), center_pix=(40, 40),
                              nsigma_mask=1.0, ncontour=9, bar_kpc=10.0,
                              fs_title=22, fs_tick=16):
    """Left: intrinsic circular speed. Right: Pa-alpha contours on the continuum.

    Follows Archive_6/1.1-astr540-plot.ipynb exactly:
      * mask = emission > 1 sigma, then ONLY the blob connected to the centre
      * the velocity field is centred on the CUTOUT centre (40, 40), not on the
        fitted x0_v/y0_v
      * |V_rot|, so a mirror-branch fit still plots positive
      * the direct image is shown with an ASINH stretch
      * contours run from 3 sigma to the peak, outermost drawn heavier
      * no tick marks; a 10 kpc bar carries the scale instead
    """
    em = np.average([_np(image_R_mc), _np(image_C_mc)], axis=0)
    direct = _np(coadd_image)
    ny, nx = em.shape

    # ---- mask: 1 sigma, connected to the centre ------------------------------
    _, _, std = sigma_clipped_stats(em)
    mask = connected_mask_from_center(em > nsigma_mask*std, center_pix)

    # ---- circular speed on the cutout-centre grid -----------------------------
    y, x = torch.meshgrid(torch.arange(ny), torch.arange(nx), indexing='ij')
    t = lambda v: torch.tensor(float(v), dtype=torch.float32)
    vcirc = arctangent_disk_rotation_speed(
        x, y, V_rot=t(abs(float(velp['V_rot']))), R_v=t(velp['R_v']),
        x0_v=t(center_pix[0]), y0_v=t(center_pix[1]),
        theta_v=t(velp['theta_v']), inc_v=t(velp['inc_v']))
    vcirc = vcirc.detach().cpu().numpy().copy()
    vcirc[~mask] = np.nan

    fig, axs = plt.subplots(1, 2, figsize=figsize)

    im = axs[0].imshow(vcirc, cmap='jet', origin='lower',
                       vmin=0, vmax=np.nanmax(np.abs(vcirc)))
    cb = fig.colorbar(im, ax=axs[0], fraction=0.046, pad=0.03)
    cb.set_label(r'$V_{\rm circ}$ [km s$^{-1}$]', fontsize=fs_tick + 2)
    cb.ax.tick_params(labelsize=fs_tick)
    axs[0].set_title('Circular speed', fontsize=fs_title, pad=8)

    bar_pix = bar_kpc/KPC_PER_PIX
    xb, yb = nx*0.05, ny*0.05
    axs[0].add_patch(patches.Rectangle((xb, yb), bar_pix, 2, color='black'))
    axs[0].text(xb + bar_pix/2, yb + 4, f'{bar_kpc:.0f} kpc', color='black',
                ha='center', va='bottom', fontsize=fs_tick)

    # ---- emission contours over the asinh-stretched continuum -----------------
    # This cutout carries a large positive sky pedestal (sigma-clipped median
    # 0.711 vs a std of 0.876), so asinh(d/std) puts the BACKGROUND at ~45% of the
    # colormap and the whole frame comes out mid-orange with no visible structure.
    # Subtract the sky first so zero maps to black, matching 1.1-astr540-plot.ipynb
    # where the input image is already background-subtracted.
    _, sky_med, sky_std = sigma_clipped_stats(direct)
    axs[1].imshow(np.arcsinh((direct - sky_med)/sky_std), cmap='gist_heat',
                  origin='lower')
    sig = sigma_clipped_stats(em)[2]
    lvl3, emax = 3.0*sig, np.nanmax(em)
    levels = [emax] if lvl3 >= emax else np.linspace(lvl3, emax, ncontour)
    axs[1].contour(em, levels=levels, colors='black', linewidths=0.9)
    axs[1].contour(em, levels=[lvl3], colors='black', linewidths=2.0)
    axs[1].set_title(r'Pa$\alpha$ emission', fontsize=fs_title, pad=8)

    # his plot_circular_speed_ax draws no centre marker; only the emission map does
    axs[1].plot(*center_pix, marker='+', color='lime', ms=12, mew=2.0)
    for ax in axs:
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlabel(''); ax.set_ylabel('')
        ax.set_aspect('equal'); ax.grid(False)
        ax.text(0.03, 0.95, f'ID: {ID}', transform=ax.transAxes,
                fontsize=fs_tick + 2, ha='left', va='top', c='white')

    plt.tight_layout()
    fig.savefig(out(fname), dpi=300, bbox_inches='tight')
    plt.show()
    print(f'saved -> {out(fname)}')
    print(f'  mask: {mask.sum()} px above {nsigma_mask:.0f} sigma, centre-connected')
    print(f'  V_circ peak inside the mask = {np.nanmax(vcirc):.1f} km/s')
    print(f'  contours: {len(levels)} levels, 3 sigma = {lvl3:.4g} to peak {emax:.4g}')
    return fig


poster_speed_and_contours(vel_mc)

In [ ]:
# Poster-ready one-dimensional circular-speed profile
# The shaded region is the MCMC 16th--84th percentile interval; it is
# limited to the centre-connected region of detected Pa$\alpha$ emission.
em = np.average([_np(image_R_mc), _np(image_C_mc)], axis=0)
_, _, em_sigma = sigma_clipped_stats(em)
em_mask = connected_mask_from_center(em > em_sigma, (40, 40))

yy, xx = np.indices(em.shape)
theta = float(vel_mc['theta_v'])
inc = float(vel_mc['inc_v'])
dx, dy = xx - 40.0, yy - 40.0
x_rot = np.cos(theta) * dx + np.sin(theta) * dy
y_rot = -np.sin(theta) * dx + np.cos(theta) * dy
r_pix = np.hypot(x_rot, y_rot / np.cos(inc))
r_max = np.nanpercentile(r_pix[em_mask], 95) * KPC_PER_PIX
r_kpc = np.linspace(0.0, r_max, 200)

# The arctangent model is fit across the full 81x81-pixel R/C cutout, but
# the plotted data-supported range is set by connected Pa-alpha emission.
fit_halfwidth_pix = 0.5 * (em.shape[0] - 1)
fit_halfwidth_kpc = fit_halfwidth_pix * KPC_PER_PIX
fit_corner_kpc = np.hypot(fit_halfwidth_kpc, fit_halfwidth_kpc)
print(f'DINGO fitted cutout: {em.shape[1]} x {em.shape[0]} pixels '
      f'(+-{fit_halfwidth_kpc:.2f} kpc along each detector axis; '
      f'{fit_corner_kpc:.2f} kpc to a corner)')
print(f'Pa-alpha-supported rotation-profile radius: {r_max:.2f} kpc '
      '(95th percentile of the centre-connected >1-sigma emission)')

i_vrot = names.index('velocity.V_rot')
i_rv = names.index('velocity.R_v')
rng = np.random.default_rng(42)
draw = rng.choice(flat.shape[0], size=min(2000, flat.shape[0]), replace=False)
v_rot = np.abs(flat[draw, i_vrot])[:, None]
r_turn = flat[draw, i_rv, None] * KPC_PER_PIX
curves = (2.0 / np.pi) * v_rot * np.arctan(r_kpc[None, :] / r_turn)
v_lo, v_med, v_hi = np.percentile(curves, [16, 50, 84], axis=0)

fig, ax = plt.subplots(figsize=(7.0, 5.2))
ax.fill_between(r_kpc, v_lo, v_hi, color='tab:blue', alpha=0.25,
                label='MCMC 16th--84th percentile')
ax.plot(r_kpc, v_med, color='tab:blue', lw=3, label='MCMC median')
ax.set(xlabel='Deprojected radius [kpc]',
       ylabel=r'$V_{\rm circ}$ [km s$^{-1}$]',
       title=f'ID {ID}: circular-speed profile')
ax.set_xlim(0, r_max)
ax.set_ylim(bottom=0)
ax.legend(frameon=False, fontsize=11)
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(out('poster_circular_speed_profile.png'), dpi=300, bbox_inches='tight')
plt.show()
print('saved ->', out('poster_circular_speed_profile.png'))

### Circular-speed profile at the two-component disk $R_{80}$

This independent poster panel combines the saved two-component F444W disk morphology with the kinematic MCMC posterior.

In [ ]:
# Use the disk R80 posterior from the two-component morphology fit, then
# evaluate independently sampled kinematic curves at those radii to get v80.
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import gammaincinv

two_comp_file = out(f'sersic_{PYSERSIC_FILTER}_pysersic_2comp.npz')
if not os.path.exists(two_comp_file):
    raise FileNotFoundError('Run the two-component F444W morphology cell first: ' + two_comp_file)

with np.load(two_comp_file) as disk_morph:
    disk_re_pix = np.asarray(disk_morph['r_eff_2']).ravel()
    disk_n = np.asarray(disk_morph['n_2']).ravel()

rng_r80 = np.random.default_rng(143)
n_draw = min(2000, len(disk_re_pix), flat.shape[0])
morph_idx = rng_r80.choice(len(disk_re_pix), size=n_draw, replace=False)
re_draw, n_sersic_draw = disk_re_pix[morph_idx], disk_n[morph_idx]
b_n = gammaincinv(2.0 * n_sersic_draw, 0.5)
r80_pix = re_draw * (gammaincinv(2.0 * n_sersic_draw, 0.8) / b_n) ** n_sersic_draw
kpc_per_arcsec = KPC_PER_PIX / arcsec_per_pixel
r80_draw = r80_pix * pixscale_mosaic * kpc_per_arcsec
r80_lo, r80_med, r80_hi = np.percentile(r80_draw, [16, 50, 84])

kin_idx = rng_r80.choice(flat.shape[0], size=n_draw, replace=False)
v80_draw = ((2.0 / np.pi) * np.abs(flat[kin_idx, i_vrot]) *
             np.arctan(r80_draw / (flat[kin_idx, i_rv] * KPC_PER_PIX)))
v80_lo, v80_med, v80_hi = np.percentile(v80_draw, [16, 50, 84])

# Extend the plotted model curve to R80, while marking any radial range
# beyond the detected Pa-alpha emission as an extrapolation.
r_plot_max = max(r_max, 1.10 * r80_hi)
r_plot = np.linspace(0.0, r_plot_max, 240)
curve_idx = rng_r80.choice(flat.shape[0], size=n_draw, replace=False)
curve_draw = ((2.0 / np.pi) * np.abs(flat[curve_idx, i_vrot])[:, None] *
              np.arctan(r_plot[None, :] /
                        (flat[curve_idx, i_rv, None] * KPC_PER_PIX)))
v_lo_disk, v_med_disk, v_hi_disk = np.percentile(curve_draw, [16, 50, 84], axis=0)

fig, ax = plt.subplots(figsize=(7.2, 5.8))
navy, crimson, r80_teal = '#003B6F', '#B11226', '#007A6E'
ax.fill_between(r_plot, v_lo_disk, v_hi_disk, color=navy, alpha=0.25,
                label='Kinematic MCMC 16th--84th percentile')
ax.plot(r_plot, v_med_disk, color=navy, lw=3, label='Kinematic MCMC median')
if r80_med > r_max:
    ax.axvspan(r_max, r_plot_max, color='0.85', alpha=0.45,
               label='beyond detected Pa$\alpha$')
ax.axvline(r80_med, color=r80_teal, ls='--', lw=2)
ax.hlines(v80_med, 0, r80_med, color=crimson, ls=':', lw=2)
ax.errorbar(r80_med, v80_med,
            xerr=[[r80_med-r80_lo], [r80_hi-r80_med]],
            yerr=[[v80_med-v80_lo], [v80_hi-v80_med]],
            fmt='o', color=crimson, capsize=3, zorder=5)
ax.annotate(r'$v_{80}=%.0f^{+%.0f}_{-%.0f}$ km s$^{-1}$' %
            (v80_med, v80_hi-v80_med, v80_med-v80_lo),
            xy=(r80_med, v80_med), xytext=(8, 22),
            textcoords='offset points', color=crimson, fontsize=11)
ax.set(xlabel='Radius in galaxy plane [kpc]',
       ylabel=r'$V_{\rm circ}$ [km s$^{-1}$]',
       title='Circular-speed profile')
ax.set_xlim(0, r_plot_max)
ax.set_ylim(bottom=0)
ax.legend(frameon=False, fontsize=10, loc='lower left')
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(out('poster_circular_speed_profile_disk_r80.png'),
            dpi=300, bbox_inches='tight')
plt.show()
print(f'disk R80 = {r80_med:.2f} -{r80_med-r80_lo:.2f} +{r80_hi-r80_med:.2f} kpc')
print(f'v80 = {v80_med:.1f} -{v80_med-v80_lo:.1f} +{v80_hi-v80_med:.1f} km/s')
print('saved ->', out('poster_circular_speed_profile_disk_r80.png'))

### Poster figure: rotation curve + emission contours

In [ ]:
# Left: MCMC circular-speed profile. Right: F444W continuum with Pa$\alpha$ contours.
# Run the preceding profile cell first; it defines r_kpc, v_lo, v_med, and v_hi.
fig, (ax_curve, ax_map) = plt.subplots(1, 2, figsize=(14.0, 6.0),
                                          gridspec_kw={'width_ratios': [1.05, 1.0]})

ax_curve.fill_between(r_kpc, v_lo, v_hi, color='tab:blue', alpha=0.25,
                      label='MCMC 16th--84th percentile')
ax_curve.plot(r_kpc, v_med, color='tab:blue', lw=3, label='MCMC median')
ax_curve.set(xlabel='Deprojected radius [kpc]',
             ylabel=r'$V_{\rm circ}$ [km s$^{-1}$]',
             title='Circular-speed profile')
ax_curve.set_xlim(0, r_max)
ax_curve.set_ylim(bottom=0)
ax_curve.legend(frameon=False, fontsize=11)
ax_curve.grid(alpha=0.25)

direct = _np(coadd_image)
_, sky_med, sky_std = sigma_clipped_stats(direct)
ax_map.imshow(np.arcsinh((direct - sky_med) / sky_std), cmap='gist_heat',
              origin='lower')
contour_sigma = sigma_clipped_stats(em)[2]
level_3sigma, em_max = 3.0 * contour_sigma, np.nanmax(em)
levels = [em_max] if level_3sigma >= em_max else np.linspace(level_3sigma, em_max, 9)
ax_map.contour(em, levels=levels, colors='black', linewidths=0.9)
ax_map.contour(em, levels=[level_3sigma], colors='black', linewidths=2.0)
ax_map.plot(40, 40, marker='+', color='lime', ms=12, mew=2.0)
ax_map.set_title(r'F444W continuum + Pa$\alpha$ contours')
ax_map.set_xticks([]); ax_map.set_yticks([])
ax_map.set_aspect('equal'); ax_map.grid(False)
ax_map.text(0.03, 0.95, f'ID: {ID}', transform=ax_map.transAxes,
            fontsize=14, ha='left', va='top', color='white')

fig.tight_layout()
fig.savefig(out('poster_rotation_curve_and_contours.png'), dpi=300, bbox_inches='tight')
plt.show()
print('saved ->', out('poster_rotation_curve_and_contours.png'))

### Add $R_{80}$ and $v_{80}$ to the poster rotation curve

In [ ]:
### This block
# v80 is V_circ evaluated at R80.  The vertical line marks R80, the
# horizontal guide marks v80, and the point is their intersection.
# R80 is derived here from the F444W single-Sersic morphology posterior.
from scipy.special import gammaincinv

if 'pysersic_post' in globals():
    re_pix = np.asarray(pysersic_post['r_eff']).ravel()
    n_sersic = np.asarray(pysersic_post['n']).ravel()
else:
    morph = np.load(out(f'sersic_{PYSERSIC_FILTER}_pysersic_notebook.npz'))
    re_pix, n_sersic = morph['r_eff'], morph['n']

morph_rng = np.random.default_rng(43)
morph_draw = morph_rng.choice(len(re_pix), size=min(2000, len(re_pix)), replace=False)
re_draw, n_draw = re_pix[morph_draw], n_sersic[morph_draw]
b_n = gammaincinv(2.0 * n_draw, 0.5)
r80_pix = re_draw * (gammaincinv(2.0 * n_draw, 0.8) / b_n) ** n_draw
kpc_per_arcsec = KPC_PER_PIX / arcsec_per_pixel
r80_draw = r80_pix * pixscale_mosaic * kpc_per_arcsec
r80_lo, r80_med, r80_hi = np.percentile(r80_draw, [16, 50, 84])

kin_draw = morph_rng.choice(flat.shape[0], size=len(r80_draw), replace=False)
v80_draw = ((2.0 / np.pi) * np.abs(flat[kin_draw, i_vrot]) *
             np.arctan(r80_draw / (flat[kin_draw, i_rv] * KPC_PER_PIX)))
v80_lo, v80_med, v80_hi = np.percentile(v80_draw, [16, 50, 84])

# Redraw the left panel far enough to include R80, retaining the right contour panel.
r_plot_max = max(r_max, 1.10 * r80_hi)
r_plot = np.linspace(0.0, r_plot_max, 240)
curve_draw = morph_rng.choice(flat.shape[0], size=min(2000, flat.shape[0]), replace=False)
profile_draw = ((2.0 / np.pi) * np.abs(flat[curve_draw, i_vrot])[:, None] *
                np.arctan(r_plot[None, :] /
                          (flat[curve_draw, i_rv, None] * KPC_PER_PIX)))
p_lo, p_med, p_hi = np.percentile(profile_draw, [16, 50, 84], axis=0)

ax_curve.clear()
navy, crimson, r80_teal = '#003B6F', '#B11226', '#007A6E'
ax_curve.fill_between(r_plot, p_lo, p_hi, color=navy, alpha=0.25,
                      label='MCMC 16th--84th percentile')
ax_curve.plot(r_plot, p_med, color=navy, lw=3, label='MCMC median')
if r80_med > r_max:
    ax_curve.axvspan(r_max, r_plot_max, color='0.85', alpha=0.45,
                     label='beyond detected Pa$\alpha$')
ax_curve.axvline(r80_med, color=r80_teal, ls='--', lw=2)
ax_curve.hlines(v80_med, 0, r80_med, color=crimson, ls=':', lw=2)
ax_curve.errorbar(r80_med, v80_med,
                  xerr=[[r80_med-r80_lo], [r80_hi-r80_med]],
                  yerr=[[v80_med-v80_lo], [v80_hi-v80_med]],
                  fmt='o', color=crimson, capsize=3, zorder=5)
ax_curve.annotate(r'$v_{80}=%.0f^{+%.0f}_{-%.0f}$ km s$^{-1}$' %
                  (v80_med, v80_hi-v80_med, v80_med-v80_lo),
                  xy=(r80_med, v80_med), xytext=(5, 14),
                  textcoords='offset points', color=crimson, fontsize=11)
ax_curve.set(xlabel='Deprojected radius [kpc]',
             ylabel=r'$V_{\rm circ}$ [km s$^{-1}$]',
             title='Circular-speed profile')
ax_curve.set_xlim(0, r_plot_max)
ax_curve.set_ylim(bottom=0)
ax_curve.legend(frameon=True, fontsize=10, loc='upper left')
ax_curve.grid(alpha=0.25)

fig.savefig(out('poster_rotation_curve_and_contours_v80.png'),
            dpi=300, bbox_inches='tight')
plt.show()
print(f'R80 = {r80_med:.2f} -{r80_med-r80_lo:.2f} +{r80_hi-r80_med:.2f} kpc')
print(f'v80 = {v80_med:.1f} -{v80_med-v80_lo:.1f} +{v80_hi-v80_med:.1f} km/s')
print('saved ->', out('poster_rotation_curve_and_contours_v80.png'))

### Overlay the projected $R_{80}$ on the Pa$\alpha$ contours

The ellipse is the disk-plane $R_{80}$ from the preceding cell, projected onto the sky using the fitted kinematic position angle and inclination.

In [ ]:
# Leave the preceding ### This block unchanged. Run it first, then run this
# cell to display that same two-panel figure with R80 overlaid on its right map.
from matplotlib.patches import Ellipse, Rectangle

if not all(name in globals() for name in ('fig', 'ax_curve', 'ax_map', 'r80_med', 'r_max',
                                          'KPC_PER_PIX', 'vel_mc', 'z')):
    raise RuntimeError('Run the two-panel poster cell and the preceding R80/v80 cell first.')

# Use the figure that owns ax_map, not the most recently created matplotlib
# figure (which may be a later one-panel rotation-curve plot).
two_panel_fig = ax_map.figure

# Remove this cell's previous overlay when it is re-run.
for artist in list(ax_map.patches) + list(ax_map.texts) + list(ax_map.lines):
    if artist.get_gid() in {'r80_map_overlay', 'rmax_map_overlay'}:
        artist.remove()

# r80_teal = '#007A6E'
r80_teal = crimson

# Use the dashed R80 guide that is already drawn in the left panel.  A later
# two-component-fit cell can overwrite the global r80_med, while this figure
# was created with the single-Sersic value from the preceding ### This block.
r80_guides = []
for line in ax_curve.lines:
    x_data = np.asarray(line.get_xdata(), dtype=float)
    if (x_data.size == 2 and np.allclose(x_data[0], x_data[1])
            and line.get_linestyle() == '--'):
        r80_guides.append(line)
if not r80_guides:
    raise RuntimeError('Could not find the dashed R80 guide in the left panel.')
r80_guide = r80_guides[-1]
r80_for_figure = float(r80_guide.get_xdata()[0])
r80_guide.set_color(crimson)

# Project that same disk-plane R80 onto the sky.
r80_major_pix = r80_for_figure / KPC_PER_PIX
rmax_major_pix = r_max / KPC_PER_PIX
q_projected = np.clip(np.cos(float(vel_mc['inc_v'])), 0.05, 1.0)
pa_deg = np.degrees(float(vel_mc['theta_v']))
# White dotted ellipse: the maximum radius supported by connected Pa-alpha
# emission. Gray dashed ellipse: the inner, morphology-derived R80.
rmax_ellipse = Ellipse((40.0, 40.0),
                       width=2.0 * rmax_major_pix,
                       height=2.0 * rmax_major_pix * q_projected,
                       angle=pa_deg, fill=False, color='white',
                       lw=3.2, ls=':', zorder=5)
rmax_ellipse.set_gid('rmax_map_overlay')
ax_map.add_patch(rmax_ellipse)

r80_ellipse = Ellipse((40.0, 40.0),
                      width=2.0 * r80_major_pix,
                      height=2.0 * r80_major_pix * q_projected,
                      angle=pa_deg, fill=False, color=r80_teal,
                      lw=3.2, ls='--', zorder=6, alpha=1)
r80_ellipse.set_gid('r80_map_overlay')
ax_map.add_patch(r80_ellipse)

# 10 kpc scale bar, matching the white rectangular style in the archived
# ASTR 540 plotting notebook. Keep the fractional pixel length for an exact
# 10 kpc bar rather than rounding it to an integer number of pixels.
map_ny, map_nx = ax_map.images[0].get_array().shape
bar_pix = 10.0 / KPC_PER_PIX
bar_x, bar_y = map_nx * 0.05, map_ny * 0.05
scale_bar = Rectangle((bar_x, bar_y), bar_pix, 2.0, color='white', zorder=8)
scale_bar.set_gid('rmax_map_overlay')
ax_map.add_patch(scale_bar)
bar_text = ax_map.text(bar_x + 0.5 * bar_pix, bar_y + 5.0, '10 kpc', color='white',
                       fontsize=10, ha='center', va='bottom', zorder=8)
bar_text.set_gid('rmax_map_overlay')

z_text = ax_map.text(0.03, 0.89, f'z = {z:.3f}', transform=ax_map.transAxes,
                     fontsize=12, ha='left', va='top', color='white', zorder=8)
z_text.set_gid('rmax_map_overlay')

# Match the circular-speed legend in the preceding R80/v80 plot block.
ax_curve.legend(frameon=True, fontsize=10, loc='upper left')

two_panel_fig.tight_layout()
two_panel_fig.savefig(out('poster_rotation_curve_and_contours_v80_with_r80_ellipse222.png'),
            dpi=300, bbox_inches='tight')
plt.figure(two_panel_fig.number)
plt.show()
print(f'projected R80 ellipse: semi-major axis = {r80_major_pix:.2f} pixels')
print('saved ->', out('poster_rotation_curve_and_contours_v80_with_r80_ellipse222.png'))

### Measured rotation curve vs the assumed arctangent

`arctangent_disk_velocity_model` hard-wires a curve that RISES MONOTONICALLY and
flattens asymptotically. It cannot produce a declining outer rotation curve, so it
answers "flat, rising, or declining?" by assumption. `rotation_curve.py` frees one
velocity per radial ring, holding the geometry fixed at the fitted values, and lets
the R--C objective determine the shape.

Two perpendicular dispersion directions are what make this possible without a light
model: requiring R and C to agree constrains the velocity locally. A single-dispersion
code cannot do it.

The module checks that the ring model reproduces the arctangent at the seed and raises
if it does not -- if the knots do not straddle `R_v`, the turnover falls inside the
first segment and the fit silently returns nonsense.

In [ ]:
import rotation_curve as rcmod

RC_RINGS = 7
rc = rcmod.fit_rotation_curve(kinematics_fitter, n_rings=RC_RINGS,
                              maxiter=300, verbose=True)

print(f'\n{"r [px]":>9}{"r [kpc]":>10}{"V_ring":>10}{"+-":>8}'
      f'{"V_arctan":>11}{"diff":>9}')
for r_, v_, e_, a_ in zip(rc['r_knots'], rc['v_knots'], rc['v_err'], rc['v_arctan']):
    print(f'{r_:>9.2f}{r_*KPC_PER_PIX:>10.2f}{v_:>10.1f}{e_:>8.1f}{a_:>11.1f}{v_-a_:>+9.1f}')

dchi2 = (rc['sse_arctan'] - rc['sse_ring'])/rc['var_base']
print(f'\nSSE  arctangent {rc["sse_arctan"]:.6f}   free rings {rc["sse_ring"]:.6f}')
print(f'delta chi2 = {dchi2:.1f} for {RC_RINGS - 2} extra parameters')
print('positive means the data prefer a shape the arctangent cannot make')

fig, ax = plt.subplots(figsize=(8.5, 6.0))
rcmod.plot_rotation_curve(rc, kpc_per_pixel=KPC_PER_PIX, ax=ax,
                          title=f'ID {ID}   measured rings vs assumed arctangent')
fig.savefig(out('rotation_curve_rings.png'), dpi=250, bbox_inches='tight')
plt.show()
np.savez_compressed(out('rotation_curve_rings.npz'), **{
    k: v for k, v in rc.items() if k != 'geometry'})
print('saved ->', out('rotation_curve_rings.npz'))

### Circular speed map beside the measured rotation curve

The same information twice: the map shows `V_circ` painted on the sky, the curve shows
it against deprojected radius with the free rings overplotted on the assumed
arctangent. Both panels are built from the SAME parameter set, printed below, so the
map and the red line are guaranteed to be the same model.

In [ ]:
# both panels use whatever velocity parameters the fitter currently holds --
# read them once and pass them to both, so the map and the curve cannot disagree
velp_now = kinematics_fitter._get_model_params('velocity')
print(f"using: V_rot={float(velp_now['V_rot']):.1f}  "
      f"inc={np.rad2deg(float(velp_now['inc_v'])):.2f} deg  "
      f"R_v={float(velp_now['R_v']):.3f} px  "
      f"(the ring fit above used these same values, geometry held fixed)")

CENTER_PIX = (40, 40)

# ---- left panel: circular speed on the sky (as in the poster figure) ---------
em_now = np.average([_np(kinematics_fitter.image_R),
                     _np(kinematics_fitter.image_C)], axis=0)
_, _, std_now = sigma_clipped_stats(em_now)
mask_now = connected_mask_from_center(em_now > 1.0*std_now, CENTER_PIX)

ny_, nx_ = em_now.shape
yy_, xx_ = torch.meshgrid(torch.arange(ny_), torch.arange(nx_), indexing='ij')
_t = lambda v: torch.tensor(float(v), dtype=torch.float32)
vcirc_now = arctangent_disk_rotation_speed(
    xx_, yy_, V_rot=_t(abs(float(velp_now['V_rot']))), R_v=_t(velp_now['R_v']),
    x0_v=_t(CENTER_PIX[0]), y0_v=_t(CENTER_PIX[1]),
    theta_v=_t(velp_now['theta_v']), inc_v=_t(velp_now['inc_v'])
).detach().cpu().numpy().copy()
vcirc_now[~mask_now] = np.nan

fig, axs = plt.subplots(1, 2, figsize=(16.5, 6.4))
im = axs[0].imshow(vcirc_now, cmap='jet', origin='lower',
                   vmin=0, vmax=np.nanmax(vcirc_now))
cb = fig.colorbar(im, ax=axs[0], fraction=0.046, pad=0.03)
cb.set_label(r'$V_{\rm circ}$ [km s$^{-1}$]', fontsize=15)
cb.ax.tick_params(labelsize=13)
axs[0].set_title('Circular speed', fontsize=18, pad=8)
bar_pix = 10.0/KPC_PER_PIX
axs[0].add_patch(patches.Rectangle((nx_*0.05, ny_*0.05), bar_pix, 2, color='black'))
axs[0].text(nx_*0.05 + bar_pix/2, ny_*0.05 + 4, '10 kpc', color='black',
            ha='center', va='bottom', fontsize=13)
axs[0].set_xticks([]); axs[0].set_yticks([])
axs[0].set_aspect('equal'); axs[0].grid(False)
axs[0].text(0.03, 0.95, f'ID: {ID}', transform=axs[0].transAxes,
            fontsize=15, ha='left', va='top', c='white')

# ---- right panel: the measured rings against the assumed arctangent ---------
rcmod.plot_rotation_curve(rc, kpc_per_pixel=KPC_PER_PIX, ax=axs[1],
                          title='Rotation curve', fs_label=15, fs_tick=13)

plt.tight_layout()
fig.savefig(out('speed_map_and_rotation_curve.png'), dpi=250, bbox_inches='tight')
plt.show()
print('saved ->', out('speed_map_and_rotation_curve.png'))

### Poster version of the R/C panel figure

Same four panels per grism as `plot.plot_kinematics_fitting_result`, restyled for print.
Three differences that matter:

- the **residual colour scale is set by the noise** (+-3 sigma of the R-C difference), not
  by the residual's own maximum. On the default scaling pure noise saturates the colormap
  and looks like structure; here anything that saturates is genuinely not noise.
- colourbars are labelled, velocity in km/s, and a kpc scale bar is drawn.
- `origin='lower'` everywhere, matching the rest of the notebook.

Note the two residual panels are the same data with the sign flipped -- row R shows
R$-$C, row C shows C$-$R -- because the objective is a single difference. Set
`DROP_DUP_RESIDUAL = True` for a 2x3 version with bigger panels.

In [ ]:
DROP_DUP_RESIDUAL = False      # True -> 2x3, drops the sign-flipped duplicate
WHICH = 'mcmc_median'          # 'adam' | 'mcmc_median' | 'mcmc_map'

_sets = {'adam':        (image_R,    image_C,    vz_R,    vz_C,    'Adam MAP'),
         'mcmc_median': (image_R_mc, image_C_mc, vz_R_mc, vz_C_mc, 'MCMC posterior median'),
         'mcmc_map':    (image_R_mp, image_C_mp, vz_R_mp, vz_C_mp, 'MCMC MAP sample')}
_iR, _iC, _vR, _vC, _tag = _sets[WHICH]
_iR, _iC, _vR, _vC = map(_np, (_iR, _iC, _vR, _vC))

resid = _np(_iR) - _np(_iC)
noise = 3*float(np.sqrt(kinematics_fitter.mcmc_var_base))
vmax  = np.nanmax(np.abs([_vR, _vC]))
ncol  = 3 if DROP_DUP_RESIDUAL else 4

fig, axs = plt.subplots(2, ncol, figsize=(4.6*ncol, 9.2))
for row, (grism, img, vz, sign) in enumerate(
        [('R', _iR, _vR, +1.0), ('C', _iC, _vC, -1.0)]):
    raw = _np(true_grism_R if grism == 'R' else true_grism_C)
    panels = [(raw,  f'{grism} grism image',        None,      None),
              (img,  f'{grism} Best-fit model', None,      None),
              (sign*resid, f'{grism} residual',     'seismic', noise),
              (vz,   f'{grism} velocity field',     'seismic', vmax)]
    if DROP_DUP_RESIDUAL and row == 1:
        panels.pop(2)
    for col, (im, title, cmap, lim) in enumerate(panels[:ncol]):
        ax = axs[row, col]
        kw = dict(origin='lower')
        if cmap: kw.update(cmap=cmap, vmin=-lim, vmax=lim)
        m = ax.imshow(np.where(nmask, im, np.nan) if 'residual' in title else im, **kw)
        ax.set_title(title, fontsize=15, pad=8)
        ax.set_axis_off(); ax.grid(False)
        cb = fig.colorbar(m, ax=ax, fraction=0.046, pad=0.04)
        cb.ax.tick_params(labelsize=10)
        if 'velocity field' in title:
            cb.set_label(r'$v_z$ [km s$^{-1}$]', fontsize=12)
        elif 'residual' in title:
            cb.set_label(r'$\pm3\sigma_{\rm noise}$', fontsize=12)
        ax.text(0.04, 0.94, f'({chr(97+row*ncol+col)})', transform=ax.transAxes,
                fontsize=13, color='white', weight='bold', va='top')

# kpc scale bar on the first panel
_bar = 5.0/KPC_PER_PIX                                  # 5 kpc in pixels
_ax = axs[0, 0]; _ny, _nx = _np(true_grism_R).shape
_ax.plot([0.06*_nx, 0.06*_nx + _bar], [0.08*_ny]*2, lw=3, color='white')
_ax.text(0.06*_nx + _bar/2, 0.10*_ny, '5 kpc', color='white', fontsize=12,
         ha='center', va='bottom')

fig.suptitle(f'ID {ID}   $z = {z:.4f}$   |   {_tag}', fontsize=17, y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(out(f'poster_panels_{WHICH}.png'), dpi=250, bbox_inches='tight')
plt.show()
print('saved ->', out(f'poster_panels_{WHICH}.png'))